# MARCIUS3R · Generación GRATIS con Wan2GP (motor local de Open-Generative-AI)
Funciona en **Kaggle** (Settings → Accelerator: GPU T4, Internet: On) o **Google Colab** (Entorno → GPU T4).

1. Ejecuta `python -m marcius3r produce` en tu PC. El vídeo queda en `voice_pending` o `scenes_pending` con dos archivos en `workspace/videos/<id>/`:
   - `generation_tasks.json`
   - `generation_manifest.json`
2. Sube **los dos** a este cuaderno. En la fase de voz, sube también `marcius3r_ref.wav`, tu voz de referencia.
3. Ejecuta todas las celdas y descarga `resultados.zip`.
4. Descomprímelo en la carpeta que indica `unzip_to`:
   - Fase de voz: `voice/`
   - Fase de escenas: `clips/`
5. Ejecuta `python -m marcius3r resume <id>`. Se repite una vez por fase: primero voz, luego escenas.


In [ ]:
!git clone -q https://github.com/deepbeepmeep/Wan2GP /tmp/Wan2GP && cd /tmp/Wan2GP && git checkout -q 38758d3c73366422efcf609abb63cab8385f3552
!cd /tmp/Wan2GP && pip install -q -r requirements.txt

In [ ]:
import json, os, glob
try:
    from google.colab import files  # Colab
    up = files.upload()
except ImportError:
    print('Kaggle: añade generation_tasks.json como Dataset o súbelo a /kaggle/working')
cands = glob.glob('/kaggle/input/**/generation_tasks.json', recursive=True) + glob.glob('generation_tasks.json') + glob.glob('/kaggle/working/generation_tasks.json')
tasks_path = os.path.abspath(cands[0]); print('tareas:', tasks_path, len(json.load(open(tasks_path))))
import shutil
for ref in glob.glob('/kaggle/input/**/marcius3r_ref.wav', recursive=True) + glob.glob('marcius3r_ref.wav'):
    shutil.copy(ref, '/tmp/Wan2GP/marcius3r_ref.wav'); print('voz de referencia copiada')


In [ ]:
!mkdir -p /tmp/out && cd /tmp/Wan2GP && python wgp.py --process {tasks_path} --output-dir /tmp/out --profile 4 2>&1 | tee /tmp/out/wgp.log | grep -E 'Task|saved|Queue completed'

In [ ]:
import re, shutil, zipfile
log = open('/tmp/out/wgp.log').read()
saved = re.findall(r'saved to Path:\s*(.+?)\s*$', re.sub(r'\x1b\[[0-9;]*m','',log), re.M)
shots = json.load(open(tasks_path))
manifest = os.path.join(os.path.dirname(tasks_path), 'generation_manifest.json')
names = [s['file'].split('/')[-1] for s in json.load(open(manifest))['shots']] if os.path.exists(manifest) else [f'result_{i:03d}.mp4' for i in range(len(saved))]
with zipfile.ZipFile('resultados.zip','w') as z:
    for src, name in zip(saved, names): z.write(src.strip(), name)
print(len(saved), 'de', len(shots), 'generados → resultados.zip')